# 🎙️ Train Whisper — Multilingual ASR (Kannada + Hindi)
**JSS SMC MCA Institute | Udaya Kumar | USN: P02ME24S126024**

### Before running:
- Go to **Runtime → Change runtime type → GPU (T4)**
- Make sure you are signed into Google Drive
- Run cells in order, top to bottom


In [ ]:
# Cell 1 — Install libraries & mount Drive
!pip install -q transformers datasets accelerate jiwer evaluate
!apt-get install -q ffmpeg

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/audio_project/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/audio_project/data',   exist_ok=True)
print('✅ Setup complete!')

In [ ]:
# Cell 2 — Download Kathbath dataset (Kannada + Hindi)
from datasets import load_dataset

print('Downloading Kathbath Kannada (~5 GB)...')
kn_dataset = load_dataset('ai4bharat/Kathbath', 'kn', trust_remote_code=True)
print('Kannada:', kn_dataset)

print('Downloading Kathbath Hindi...')
hi_dataset = load_dataset('ai4bharat/Kathbath', 'hi', trust_remote_code=True)
print('Hindi:', hi_dataset)

In [ ]:
# Cell 3 — Preprocess audio for Whisper (resample to 16kHz)
from transformers import WhisperFeatureExtractor, WhisperTokenizer

MODEL_NAME = 'openai/whisper-small'
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=None, task='transcribe')

def preprocess_sample(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features[0]
    batch['labels'] = tokenizer(batch['sentence']).input_ids
    return batch

print('Preprocessing Kannada (10-30 min)...')
kn_proc = kn_dataset['train'].map(preprocess_sample,
    remove_columns=kn_dataset['train'].column_names, num_proc=2)

print('Preprocessing Hindi...')
hi_proc = hi_dataset['train'].map(preprocess_sample,
    remove_columns=hi_dataset['train'].column_names, num_proc=2)

kn_proc.save_to_disk('/content/drive/MyDrive/audio_project/data/kn_processed')
hi_proc.save_to_disk('/content/drive/MyDrive/audio_project/data/hi_processed')
print('✅ Saved to Drive!')

In [ ]:
# Cell 4 — Data Collator
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch
from transformers import WhisperProcessor
from datasets import load_from_disk, concatenate_datasets

kn_data = load_from_disk('/content/drive/MyDrive/audio_project/data/kn_processed')
hi_data = load_from_disk('/content/drive/MyDrive/audio_project/data/hi_processed')

# Combine Kannada + Hindi
combined = concatenate_datasets([kn_data, hi_data]).shuffle(seed=42)
split = combined.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(split["train"])} | Eval: {len(split["test"])}')

MODEL_NAME = 'openai/whisper-small'
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=None, task='transcribe')

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

print('✅ Collator ready!')

In [ ]:
# Cell 5 — Load Whisper model & define training
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = None
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)

metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,   skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids,  skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

OUTPUT_DIR = '/content/drive/MyDrive/audio_project/models/whisper-kn-hi'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,           # Increase to 2000-4000 for better accuracy
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy='steps',
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,
    eval_steps=200,
    logging_steps=50,
    report_to=['tensorboard'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor
)

print('🚀 Starting Whisper fine-tuning (~1.5-2 hrs on T4)...')
trainer.train()
print('✅ Training complete! Model saved to Drive.')

In [ ]:
# Cell 6 — Zip & download trained model
import shutil
from google.colab import files

print('Zipping model...')
shutil.make_archive('/content/whisper_kn_hi', 'zip',
                    '/content/drive/MyDrive/audio_project/models/whisper-kn-hi')
print('Downloading...')
files.download('/content/whisper_kn_hi.zip')
print('✅ Extract the zip to: audio_notes_project/models/whisper-kn-hi/')